# CRISP-DM Fase 4: Data Modeling - Arquitectura Relacional y Lakehouse
## Dataset: SIPSA Precios de Insumos y Fertilizantes Agrícolas (DANE)

**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 4: Dimensional & Relational Modeling) | **Fase PDCO**: DEVELOPMENT  
**Estándares**: Medallion Architecture (Gold Layer), Kimball Dimensional Modeling, SQL ACID Compliance.

---

### 🏛️ 1. Diseño Relacional y Dimensional (Lakehouse Gold)
* Creación de tabla relacional en el Lakehouse estructurado (`agrostats_lakehouse.db`).
* Verificación de claves primarias, tipos SQL conformados e indexación para consultas analíticas de alta velocidad.
* Verificación de consistencia cruzada entre el almacenamiento columnar Parquet y el motor relacional SQL.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


In [ ]:
# ==============================================================================
# [FASE 6: CARGA IDEMPOTENTE AL DATA LAKEHOUSE (SQLITE GOLD)]
# ==============================================================================
from src.database.db_manager import DatabaseManager

db_path = APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"
db = DatabaseManager(str(db_path))

parquet_file = APP_ROOT / "data/processed/sipsa_insumos.parquet"
df = pd.read_parquet(parquet_file)

table_name = "sipsa_insumos"
print(f"🗄️ Persistiendo {len(df):,} registros en tabla SQL '{table_name}'...")
db.save_dataframe(df, table_name, if_exists="replace")

# Inspección de esquema SQL
schema_info = db.query(f"PRAGMA table_info('{table_name}')")
print("\n📋 Esquema Relacional de la Tabla:")
display(schema_info)


In [ ]:
# ==============================================================================
# [FASE 6: CONSULTAS ANALÍTICAS Y CONTROL DE INTEGRIDAD PARQUET vs SQL]
# ==============================================================================
primary_col = "indice_total"
group_col = "grupo_producto"

# Query analítica de agregación
if primary_col in df.columns and pd.api.types.is_numeric_dtype(df[primary_col]):
    query = "SELECT COUNT(*) AS total_registros, ROUND(AVG(indice_total), 2) AS promedio, ROUND(MIN(indice_total), 2) AS minimo, ROUND(MAX(indice_total), 2) AS maximo FROM sipsa_insumos"
    agg_res = db.query(query)
    print("📈 Resumen Agregado SQL:")
    display(agg_res)

# Verificación de paridad de registros (Parquet == SQL)
count_sql = db.query("SELECT COUNT(*) AS c FROM sipsa_insumos").iloc[0]['c']
count_parquet = len(df)
assert count_sql == count_parquet, f"Error: Desalineación entre Parquet ({count_parquet}) y SQL ({count_sql})"
print(f"✅ Integridad verificada al 100%: {count_sql:,} registros consistentes en Lakehouse.")
